# 79장 · JavaScript 폼 이벤트와 fetch 응답 처리

과제 화면의 제출 버튼을 눌렀을 때 서버가 받을 값과 화면에 표시할 문장을 하나씩 따라갑니다.

과정 안내 화면의 submit 이벤트에서 요청 JSON을 만들고, HTTP 응답을 결과 영역에 반영합니다. 노트북에서는 이벤트와 응답을 재현해 정상·오류 경로를 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## submit 이벤트와 JSON 요청 본문

브라우저의 submit 처리기는 event.preventDefault()로 기본 페이지 이동을 막고 입력값의 양끝을 정리합니다. 노트북에서는 Node.js가 설치된 수업 커널에서 실제 JavaScript를 실행합니다. DOM을 띄우지 않고 표준 EventTarget을 이용해 제출 동작만 재현합니다.


### 제출 이벤트가 읽는 입력값

<p>submit 이벤트에서 preventDefault()로 기본 폼 전송을 멈추고 입력값의 양끝 공백을 trim()으로 제거합니다. JSON.stringify가 만든 문자열을 보면 question은 문자열이고 k는 숫자입니다.</p>


### 셀 01 · JavaScript 실행 환경 준비

코드를 바꿀 곳은 다음 셀의 JavaScript입니다. 제공 함수는 실행 결과를 노트북에 표시합니다.


In [1]:
import subprocess
def run_js(source):
    run = subprocess.run(["node", "--input-type=module", "-e", source],
                         text=True, capture_output=True, check=True, timeout=10)
    print(run.stdout.rstrip())

함수를 정의했습니다. 별도의 JS 파일을 만들 필요가 없습니다.


## fetch 요청 본문에 선택한 ID 담기

폼에서 두 번째 항목을 선택하면 해당 후보만 검토 대상으로 올라가야 합니다. fetch에 넘길 JSON을 구성하고 서버에서 숫자 ID를 받는지 확인합니다.

노트북을 같은 수업 폴더에 모아 열거나 압축을 푼 장별 폴더에서 실행하세요. 시작 셀은 service-lab/form.json을 먼저 엽니다. 파일이 없을 때만 셀에 적힌 복귀 예시를 읽습니다.

In [ ]:
from pathlib import Path
import json
here = Path.cwd()
# 각 장 폴더에서 열거나 노트북을 한 폴더에 모아 두어도 같은 실습 폴더를 씁니다.
course = here.parent if here.name in ("daily",) or (here.name.startswith("chapter") and here.name[7:].isdigit()) else here
lab = course / "service-lab"
lab.mkdir(exist_ok=True)
incoming = course / "service-lab/form.json"
if incoming.exists():
    previous = json.loads(incoming.read_text(encoding="utf-8"))
    print("내 실습 파일", incoming.name)
else:
    previous = {'items': [{'id': 1,
                'title': '변경 검토',
                'candidate': '42ec606abc92ce832234dded6ceb188e2a0f771365698f2b6d2215d7be9aebb2'},
               {'id': 2, 'title': '변경 검토', 'candidate': 'revision-b'}],
     'values': ['1', '2']}
    print("복귀 예시", incoming.name)

내 실습 파일 form.json


### fetch로 선택한 ID 전송하기

In [ ]:
import json, subprocess
from fastapi import FastAPI
from fastapi.testclient import TestClient
selection = previous["values"][-1]
js = "const id = Number(process.argv[1]); console.log(JSON.stringify({id, done:true}));"
body = json.loads(subprocess.check_output(["node", "-e", js, selection], text=True))
preview = FastAPI()
@preview.post("/preview")
def preview_change(change: dict):
    item = next(row for row in previous["items"] if row["id"] == change["id"])
    return {"id": item["id"], "candidate": item["candidate"], "done": change["done"]}
reply = TestClient(preview).post("/preview", json=body)
reply.raise_for_status()
result = {"items": previous["items"], "proposal": reply.json()}
print(body, reply.json()["candidate"])

{'id': 2, 'done': True} revision-b


select.value는 문자열입니다. Number를 거친 ID가 JSON 숫자로 전송되므로 서버에서 2번 항목을 찾습니다. preview는 제안만 반환합니다. 실제 저장은 다음 장의 승인 단계에서 확인합니다.

### 직접 해 보기

1번과 2번의 표시 제목이 같을 때 제안에 들어간 후보가 실제 둘째 항목의 후보인지 대조하세요.

In [ ]:
# 응답의 id와 candidate를 둘째 항목과 각각 비교하세요.

<details class="notebook-answer"><summary>정답 코드와 실행 결과</summary><pre><code>second = previous[&quot;items&quot;][1]
print(result[&quot;proposal&quot;][&quot;id&quot;] == second[&quot;id&quot;], result[&quot;proposal&quot;][&quot;candidate&quot;] == second[&quot;candidate&quot;])</code></pre><pre>True True</pre><p>같은 제목의 첫 행을 고르면 ID와 후보 값이 모두 달라집니다.</p></details>

### 80장에서 다시 열 파일

방금 계산한 값을 service-lab/selection.json에 저장합니다. 같은 장에서 조건을 바꾸고 다시 저장하면 다음 장도 그 값을 읽습니다.

In [ ]:
destination = lab / "selection.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination.name)

selection.json


### 셀 02 · submit 이벤트의 입력값

질문 양끝에 있는 공백이 JSON 본문에 남을지 예측하세요.


In [2]:
run_js("""
const form = new EventTarget();
const input = {value: "  과제는 어디에 내나요?  "};
form.addEventListener("submit", event => {
  event.preventDefault();
  const body = {question: input.value.trim(), k: 2};
  console.log(JSON.stringify(body));
});
form.dispatchEvent(new Event("submit", {cancelable: true}));
""")

{"question":"과제는 어디에 내나요?","k":2}


출력된 JSON의 question은 양끝 공백을 뺀 문자열이고 k는 숫자 2입니다. 이 셀은 요청 본문을 만드는 단계에서 끝납니다. 다음 셀에서 응답을 만들고 표시할 답변을 읽습니다.


#### 응답 본문이 있는데 화면이 실패할 때

<mark>요청을 보낸 값과 응답에서 읽어야 할 필드 이름을 각각 확인합니다.</mark> 입력 이벤트가 올바른 JSON을 만들었다면 다음에는 서버가 보낸 답변·출처를 보세요. 응답을 받은 시점과 그 값을 화면에 반영한 시점 사이에서도 오류가 생깁니다.

선수 개념 · 68장 FastAPI 요청 모델과 POST 엔드포인트

### 대화 경로와 질문 본문의 역할

<p>예제의 경로에는 chat-a가, JSON 본문에는 question과 k가 들어갑니다. 경로는 어느 대화로 보낼지를, 본문은 이번에 무엇을 물을지를 나타냅니다. 이 셀은 네트워크에 접속하지 않고 fixtureFetch로 응답을 만듭니다.</p>

<h3>chat-a의 모의 응답에서 읽는 값</h3>

<p>셀 03은 대화가 이미 있다는 가정 아래 /conversations/chat-a/ask를 사용합니다. fixtureFetch가 질문을 읽고 result.answer를 돌려주면 예제 객체의 textContent에 넣습니다. 대화 생성이나 DB 저장을 실행하는 셀은 없습니다.</p>


## 비동기 응답과 오류 분기

실제 화면의 transport.mjs는 fetch 함수를 전달받습니다. 이 셀에서는 수업용 응답을 주는 함수를 넣어 요청과 화면 사이를 재현합니다. 실제 서버 주소에서는 해당 함수 자리에 브라우저 fetch를 넣습니다.


### 응답 도착과 JSON 읽기의 두 await

<p>첫 await는 모의 함수가 돌려주는 Response를 기다립니다. 성공 상태를 확인한 뒤 두 번째 await로 JSON 본문을 읽습니다. 응답 객체 자체에는 아직 result.answer 형태의 속성이 없습니다.</p>


### 셀 03 · 응답 JSON을 모의 객체의 textContent에 대입

fetch 응답과 JSON 해석을 각각 await합니다. 여기서 element는 화면 요소의 textContent 속성만 흉내 낸 객체입니다.


In [3]:
run_js("""
const fixtureFetch = async (_url, options) => {
  const { question } = JSON.parse(options.body);
  if (question !== "과제는 어디에 내나요?") return new Response(null, {status: 422});
  return Response.json({result: {answer: "과제함에 제출하세요. [course-guide-v1:p2]"}});
};
const response = await fixtureFetch("/conversations/chat-a/ask", {
  method: "POST", headers: {"Content-Type":"application/json"},
  body: JSON.stringify({question:"과제는 어디에 내나요?", k:2})
});
if (!response.ok) throw Error("HTTP " + response.status);
const body = await response.json();
const element = {textContent: ""};
element.textContent = body.result.answer;
console.log(element.textContent);
""")

과제함에 제출하세요. [course-guide-v1:p2]


표시할 텍스트에는 출처 ID가 남았습니다. innerHTML로 문자열을 해석하지 않고 textContent에 넣으면 답에 꺾쇠가 있더라도 문구로 다룰 수 있습니다.


#### 모의 POST 응답에서 답변 텍스트가 되는 경로

```mermaid
flowchart TD
nsend["POST"]
  ncheck["질문 확인"]
  nok["응답 확인"]
  njson["JSON 읽기"]
  nrender["텍스트 표시"]
  nreject["422 오류"]
  nsend --> ncheck
  ncheck --> nok
  ncheck --> nreject
  nok --> njson
  njson --> nrender
```

앞 셀의 질문을 넣은 성공 경로입니다. 질문이 달라 fixtureFetch가 422를 반환하면 본문 표시 단계에 도달하지 않습니다.

### result.answer에서 textContent까지

<p>답변 문자열은 응답 맨 위가 아니라 result 안에 있습니다. 예제는 이 문자열을 textContent에 대입한 뒤 출력합니다. 실제 DOM에서 textContent에 대입해도 문자열을 HTML로 해석하지 않습니다.</p>


### 과제 04 · HTTP 실패 시 분기

본문이 빈 404 응답을 받더라도 JSON 파싱 오류가 나지 않게 처리하세요. 앞 셀의 성공 처리와 같은 위치에서 상태 코드를 먼저 읽고 안내 문구를 출력하세요.


In [4]:
# Node의 Response(null, {status: 404})를 사용해 상태별 처리를 작성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>run_js(&quot;&quot;&quot;
const response = new Response(null, {status: 404});
if (response.status === 404) {
  console.log(&quot;대화를 찾을 수 없습니다&quot;);
} else if (response.ok) {
  const body = await response.json();
  console.log(body.result.answer);
} else {
  console.log(&quot;HTTP &quot; + response.status);
}
&quot;&quot;&quot;)</code></pre>
<p>실행 결과</p><pre><code>대화를 찾을 수 없습니다</code></pre>
<p>빈 본문에 response.json()을 먼저 호출하면 예외가 나서 안내 문구까지 가지 못합니다. 여기서는 Response를 직접 만들었습니다. 실제 연결이 끊겨 응답 객체 자체를 받지 못한 경우는 별도의 예외 처리 대상입니다.</p>
</details>


### 404 응답을 받았을 때 표시할 문구

<p>404인 Response는 빈 본문을 가질 수 있습니다. 이 경우에는 상태별 안내를 먼저 선택해 JSON 파싱 오류가 실제 HTTP 상태를 가리지 않게 합니다.</p>


## 요청이 끝나기 전에 다시 눌린 제출 버튼 처리하기

입력 이벤트에서 fetch를 호출하는 흐름을 실제 서비스에 연결하면 첫 응답 전에 두 번째 제출이 들어올 수 있습니다. 응답 시점을 직접 정할 수 있는 비동기 대역을 써서 버튼과 전송 횟수를 검산합니다.

### 셀 06 · 두 이벤트가 같은 응답을 기다릴 때 호출 수 세기

셀 01에서 run_js를 정의하고 Node가 설치된 커널에서 진행하세요. 대기 중인 Promise를 풀기 전 submit을 두 번 호출하면 요청은 몇 번 시작될까요?

In [5]:
pending_js = """
let release;
let calls = 0;
const pending = new Promise(resolve => { release = resolve; });
async function send() { calls += 1; return pending; }
"""
run_js(pending_js + """
async function submit() { return send(); }
const first = submit();
const second = submit();
console.log(String(calls));
release(new Response(null, {status: 204}));
await Promise.all([first, second]);
""")

2


2가 나옵니다. 두 호출 모두 첫 번째 await에 도달하기 전에 send를 실행했습니다. 화면에 ‘전송 중’ 문구를 표시하는 것만으로는 두 번째 이벤트를 중단할 수 없습니다.

### 과제 07 · 요청을 한 번 보내고 오류 후 다시 입력할 수 있게 하기

같은 pending_js에서 submit을 구현하세요. 이미 전송 중이면 반환하고, 첫 요청은 send 전에 잠그세요. 서버가 422로 응답해도 finally에서 잠금을 풀고 완료 후 호출 횟수·상태를 출력하세요.

In [6]:
# pending_js 뒤에 버튼 잠금과 submit 함수를 작성해 run_js로 실행하세요.
# 응답을 해제하기 전 submit을 두 번 호출하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>run_js(pending_js + &quot;&quot;&quot;
let sending = false;
let message = &quot;&quot;;
async function submit() {
  if (sending) return;
  sending = true;
  try {
    const response = await send();
    message = response.ok ? &quot;저장됨&quot; : &quot;입력을 확인하세요&quot;;
  } finally {
    sending = false;
  }
}
const first = submit();
const second = submit();
release(new Response(null, {status: 422}));
await Promise.all([first, second]);
console.log([calls, sending, message].join(&quot; &quot;));
&quot;&quot;&quot;)</code></pre>
<p>실행 결과</p><pre><code>1 false 입력을 확인하세요</code></pre>
<p>1 false 입력을 확인하세요가 나옵니다. 두 번째 호출은 sending을 읽고 돌아갔고 첫 호출이 422를 읽은 뒤 잠금을 풀었습니다. 실제 폼에서는 sending 값에 맞춰 버튼의 disabled 속성과 상태 문구를 갱신하세요.</p>
</details>

<mark>전송 중 표시는 첫 await보다 먼저 바꾸세요.</mark> await send() 뒤에서 잠그면 기다리는 동안 들어온 두 번째 이벤트도 send를 호출합니다. 이 검사는 브라우저의 한 폼 안에서 중복 요청을 줄이는 방법을 다룹니다.

다음 장의 투두 추가 버튼에 같은 상태 전환을 붙인 뒤 정상 응답과 오류 응답에서 모두 버튼을 다시 쓸 수 있는지 확인하세요.

## 돌아보기

제출 이벤트의 입력에는 양끝 공백이 있고 JSON 요청에는 없습니다. 응답이 실패했을 때 입력을 다시 보내기 전에는 어느 대화 ID와 질문이 요청에 실렸는지 확인합니다.
